In [ ]:
%%time
### cell 0 ###

import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%time
### cell 1 ###

import os

def load_orders(data_folder: str, **storage_options) -> pd.DataFrame:
    # Build the path and read CSV in one step with date parsing
    file_path = os.path.join(data_folder, "orders.tbl")
    return pd.read_csv(
        file_path,
        sep="|",
        parse_dates=["O_ORDERDATE"],
        date_parser=lambda x: pd.to_datetime(x, format="%Y-%m-%d"),
        **storage_options,
    )

orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%time
### cell 0 ###

keys = lineitem.loc[
    lineitem.L_COMMITDATE < lineitem.L_RECEIPTDATE,
    "L_ORDERKEY"
].unique()
matched_orders = orders.loc[
    orders.O_ORDERDATE.between(pd.Timestamp("1993-08-01"), pd.Timestamp("1993-11-01"), inclusive="left") &
    orders.O_ORDERKEY.isin(keys),
    :
]

In [ ]:
%%time
### cell 1 ###

total = (
    matched_orders
    .groupby("O_ORDERPRIORITY", as_index=False)["O_ORDERKEY"].count()
    # skip sort when Mars enables sort in groupby
)